# Nasdaq-100: sliding-window RP computation benchmark

**Data:** Nasdaq-100 index (`^NDX`), 5-minute regular-session observations from the **last 60 calendar days** at execution time. **Window sizes:** 25, 50, 100, 250, 500, 1,000. Compare Euclidean with symmetric β-divergence for **β = 1, 3, 7**. Every eligible overlapping window is processed, moving forward one observation at a time. Results report average **seconds per thresholded RP**, not total time per window size.

## Rolling data window

Fetch `period="60d", interval="5m"` with yfinance, within Yahoo's approximately 60-day intraday retention window. These are calendar days, not 60 trading sessions; weekends and market holidays contribute no regular-session bars. Print the actual returned dates and observation count. No local archive is required.

[yfinance intraday history documentation](https://ranaroussi.github.io/yfinance/reference/yfinance.price_history.html). Yahoo availability and download success can vary; missing data are not replaced with another period, frequency, ETF, or synthetic observations.

**No download or benchmark has been executed in this notebook.** Requires NumPy, pandas, yfinance, and Jinja2 for the final styled table. No RP rendering is performed during benchmarking.

In [16]:
from time import perf_counter
import platform
import sys

import numpy as np
import pandas as pd
import yfinance as yf

TICKER = "^NDX"
LOOKBACK_DAYS = 35
INTERVAL = "5m"
WINDOW_SIZES = (25, 50, 100, 250, 500, 1000)
METRICS = ("Euclidean", "beta=1", "beta=3", "beta=7")
RR_TARGET = 0.10

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| yfinance:", yf.__version__)

Python: 3.13.9 | Platform: Windows-11-10.0.26200-SP0
NumPy: 2.4.6 | pandas: 2.3.3 | yfinance: 1.7.0


## 1. Fetch the latest 60 days

Use live yfinance data, filtered to the rolling 60-calendar-day window and regular-session timestamps in New York. No prices are imputed: invented returns would alter the recurrence workload. Invalid closes and duplicate timestamps are counted and removed. Non-5-minute inputs or insufficient observations stop the notebook.

In [17]:
request_end = pd.Timestamp.now(tz="UTC")
request_start = request_end - pd.Timedelta(days=LOOKBACK_DAYS)
history = yf.Ticker(TICKER).history(
    period=f"{LOOKBACK_DAYS}d", interval=INTERVAL,
    prepost=False, auto_adjust=False, actions=False,
    repair=False, keepna=True, raise_errors=True,
)
if history.empty or "Close" not in history:
    raise RuntimeError("Yahoo returned no ^NDX 5-minute closes for the last 60 days")
if not isinstance(history.index, pd.DatetimeIndex) or history.index.tz is None:
    raise ValueError("Input must have timezone-aware intraday timestamps")

close = pd.to_numeric(history["Close"], errors="coerce").astype(float)
close.index = close.index.tz_convert("America/New_York")
close = close.sort_index()
duplicate_count = int(close.index.duplicated(keep="last").sum())
close = close.loc[~close.index.duplicated(keep="last")]
close = close.loc[(close.index >= request_start) & (close.index < request_end)]
close = close.between_time("09:30", "16:00", inclusive="left")
invalid_count = int((~np.isfinite(close) | (close <= 0)).sum())
close = close.loc[np.isfinite(close) & (close > 0)].rename("NDX_CLOSE")
if close.empty:
    raise ValueError("No valid regular-session closes remain in the last 60 days")
if np.any(close.index.minute % 5 != 0) or np.any(close.index.second != 0):
    raise ValueError("Timestamps are not on a five-minute grid")
steps = close.index.to_series().diff().dropna()
if steps.empty or steps.mode().iloc[0] != pd.Timedelta(minutes=5):
    raise ValueError("Input is not predominantly five-minute observations")

print(f"Source: yfinance; requested rolling window {request_start} to {request_end}")
print(f"Clean closes: {len(close):,}; {close.index[0]} to {close.index[-1]}")
print(f"Duplicate timestamps removed: {duplicate_count}; invalid closes removed: {invalid_count}")
print("Observed valid closes per month (review for gaps):")
print(close.groupby(close.index.tz_localize(None).to_period("M")).size().to_string())

<cell In[17]>:3: DeprecationWarning: 'raise_errors' deprecated, do: yf.config.debug.hide_exceptions = False
  history = yf.Ticker(TICKER).history(


Source: yfinance; requested rolling window 2026-09-16 22:08:52.996195+00:00 to 2026-10-21 22:08:52.996195+00:00
Clean closes: 1,950; 2026-09-17 09:30:00-04:00 to 2026-10-21 15:55:00-04:00
Duplicate timestamps removed: 0; invalid closes removed: 0
Observed valid closes per month (review for gaps):
Datetime
2026-09     780
2026-10    1170
Freq: M


## 2. Prepare comparable positive inputs

Use **gross five-minute returns** $g_t=P_t/P_{t-1}=\exp(r_t)$, where $r_t$ is the decimal log return. All four methods use the **same** input values and scalar states (embedding dimension 1), isolating divergence-computation cost rather than mixing representations. This differs from the z-scored Euclidean inputs in the earlier analysis notebooks.

Retain a return only when its two prices are exactly five minutes apart. Overnight, weekend, and missing-bar returns are excluded. Windows contain consecutive **retained observations**, so they can cross session breaks; their length is an observation count, not uninterrupted wall-clock minutes. Incomplete windows are not used.

In [18]:
five_minute_pair = close.index.to_series().diff().eq(pd.Timedelta(minutes=5))
log_returns = np.log(close).diff().loc[five_minute_pair]
gross_returns = np.exp(log_returns)
values = np.ascontiguousarray(gross_returns.to_numpy(dtype=np.float64))
if not np.isfinite(values).all() or np.any(values <= 0):
    raise ValueError("Gross returns must be finite and strictly positive")
if len(values) < max(WINDOW_SIZES):
    raise ValueError(f"Need at least {max(WINDOW_SIZES)} valid returns; got {len(values)}")

workload = pd.DataFrame({
    "window_size": WINDOW_SIZES,
    "windows": [len(values) - size + 1 for size in WINDOW_SIZES],
})
print(f"Retained five-minute returns: {len(values):,}")
print(f"Excluded non-five-minute transitions: {len(close) - 1 - len(values):,}")
print(workload.to_string(index=False))
print(f"Total thresholded RPs to compute: {len(METRICS) * int(workload['windows'].sum()):,}")

Retained five-minute returns: 1,925
Excluded non-five-minute transitions: 24
 window_size  windows
          25     1901
          50     1876
         100     1826
         250     1676
         500     1426
        1000      926
Total thresholded RPs to compute: 38,524


## 3. What is timed?

Each call constructs the **full square divergence/distance matrix**, selects ε as the 10th percentile of off-diagonal upper-triangle entries, and produces the full Boolean thresholded RP. Matrix allocation, per-window β transforms, quantile selection, and thresholding are included. Data fetching/cleaning, return preparation, window slicing, shared upper-triangle index preparation, warm-up calls, reporting, and Matplotlib rendering are excluded.

For scalar states, Euclidean distance is $|x-y|$. As in the previous notebooks, use the symmetric mean of forward and reverse β-divergences:

$$
S_1(x,y)=\tfrac12(x-y)(\log x-\log y),\qquad
S_\beta(x,y)=\frac{(x-y)(x^{\beta-1}-y^{\beta-1})}{2(\beta-1)}.
$$

[β-divergence definition: Févotte & Idier, §2.1](https://arxiv.org/abs/1010.1763). Thresholds are fitted separately for each method and window; they have different units. Ties may make actual recurrence exceed the target. Symmetrization is included for β methods. Only one returned RP is kept at a time; no collection of window matrices is stored.

In [19]:
def build_thresholded_rp(window, metric, upper, target_rr):
    distance = np.subtract.outer(window, window)
    if metric == "Euclidean":
        np.abs(distance, out=distance)
    else:
        beta = int(metric.split("=")[1])
        transformed = np.log(window) if beta == 1 else window ** (beta - 1) / (beta - 1)
        distance *= np.subtract.outer(transformed, transformed)
        distance *= 0.5
    # Advanced indexing creates a temporary vector; quantile may reorder that copy.
    epsilon = np.quantile(distance[upper], target_rr, overwrite_input=True)
    return distance <= epsilon

## 4. Run all sliding windows — computationally intensive

**This cell performs the benchmark.** For every requested size, move forward by one retained observation and process **every** full window with all four methods. Nothing is subsampled. Each window produces a new RP from scratch; this does not benchmark an incremental-update algorithm.

A small, untimed warm-up precedes the measurements. Method order rotates between windows to reduce systematic first/last-method bias. Each RP is timed once with `perf_counter`; the average is over all windows of that size. Results depend on CPU, NumPy version, memory pressure, and other running processes. Overlapping windows are not independent statistical replicates. Keep the machine otherwise idle.

The largest windows require quadratic memory per matrix. Rendering thousands of images would measure plotting overhead and is deliberately omitted.

In [20]:
# Small untimed warm-up; no benchmark or download runs before you execute these cells.
warm_size = min(WINDOW_SIZES)
warm_upper = np.triu_indices(warm_size, k=1)
for metric in METRICS:
    warm_rp = build_thresholded_rp(values[:warm_size], metric, warm_upper, RR_TARGET)
    del warm_rp

rows = []
for size in WINDOW_SIZES:
    upper = np.triu_indices(size, k=1)  # shared by every method/window of this size
    n_windows = len(values) - size + 1
    elapsed = {metric: 0.0 for metric in METRICS}
    for start in range(n_windows):
        window = values[start : start + size]  # view; not timed
        shift = start % len(METRICS)
        order = METRICS[shift:] + METRICS[:shift]
        for metric in order:
            t0 = perf_counter()
            rp = build_thresholded_rp(window, metric, upper, RR_TARGET)
            seconds = perf_counter() - t0
            elapsed[metric] += seconds
            del rp  # release before the next call; never retain all window RPs
    for metric in METRICS:
        rows.append({
            "window_size": size,
            "metric": metric,
            "windows": n_windows,
            "total_seconds": elapsed[metric],
            "mean_seconds_per_rp": elapsed[metric] / n_windows,
        })
    print(f"Completed size {size:,}: {n_windows:,} windows for each method")
    del upper

benchmark = pd.DataFrame(rows)
print("Benchmark complete.")

Completed size 25: 1,901 windows for each method
Completed size 50: 1,876 windows for each method
Completed size 100: 1,826 windows for each method
Completed size 250: 1,676 windows for each method
Completed size 500: 1,426 windows for each method
Completed size 1,000: 926 windows for each method
Benchmark complete.


## 5. Average seconds per recurrence plot

Each entry is **total measured construction time / number of sliding windows** for that size and method. Download time, data preparation, and rendering are not included. No timing values are pre-filled. `benchmark` also retains the window counts and measured totals for inspection.

In [33]:
average_seconds = benchmark.pivot(
    index="window_size", columns="metric", values="mean_seconds_per_rp",
).reindex(index=WINDOW_SIZES, columns=METRICS)
average_seconds.index.name = "RP size (N × N)"
average_seconds.columns.name = "Mean seconds per thresholded RP"
average_seconds.style.format("{:.6f}")

Mean seconds per thresholded RP,Euclidean,beta=1,beta=3,beta=7
RP size (N × N),,,,
25,0.000033,0.000037,0.000038,0.000038
50,0.000043,0.000048,0.000048,0.000050
100,0.000078,0.000088,0.000089,0.000092
250,0.000323,0.000365,0.000369,0.000369
500,0.001998,0.002730,0.002734,0.002730
1000,0.008325,0.011158,0.011165,0.011159
